# SoS forward-model handoff

Loads final saved posterior particles. **No sampling or optimization is called.**
All results are conditional on the frozen dates, ML-selected signs, prior supports
and total-population denominator. Completion of tempering is not proof of convergence.

Extract the standalone archive and open this notebook from its top-level folder.
Select a case below, or use `all`. All 4,000 particles are retained.


In [ ]:
from pathlib import Path
import json
import numpy as np
from handoff_tools import regenerate, load_case, available_cases, QUANTILES

DATA_ROOT = Path.cwd()  # Standalone archive root, containing cases/ and sumsigmoid/
CASE = "all"           # "case1", "case2" (seed 106), "case2_seed107", "case2_seed108", "toronto", or "all"
OUTPUT = DATA_ROOT / "regenerated"
selected = available_cases(DATA_ROOT) if CASE == "all" else [CASE]
print("Available training-cleared cases:", selected)
if (DATA_ROOT / "STATUS.json").exists():
    print((DATA_ROOT / "STATUS.json").read_text())
assert all(c in available_cases(DATA_ROOT) for c in selected), "Case unavailable; read STATUS.json"


## Reconstruct and verify

For each saved parameter vector, the shared inference Euler kernel propagates
continuously from day 0 through day 199.9. No future observations enter the
likelihood or update states. The training data remain days 0–159.
Predictive noise uses a separate recorded local seed; no inference RNG is used.

Reference checks use atol=1e-7 and rtol=1e-12 for arrays/intervals/diagnostic
metrics. All-particle Gaussian-mixture quantiles are distinct from empirical
quantiles of the finite observation-noise draws. No WIS is reconstructed here.


In [ ]:
results = {}
for case in selected:
    results[case] = regenerate(DATA_ROOT, case, OUTPUT, verify=True)
    metrics = results[case]["metrics"]
    print(case, results[case]["verification"]["status"],
          "training RMSE (observed days 0–159):", metrics["training_RMSE_observed_days0_159"],
          "held-out daily RMSE:", metrics["heldout_RMSE_daily_days160_199"])
    print("Figures and arrays:", OUTPUT / case)


## Arrays for Brandon's scoring and figure code

Arrays have axes **(draw, time)**. CSV particles have draws in rows with named
parameters. `parameter_order.json` maps every amplitude to its fixed activation
date and records the model's actual column order and units.

`latent_predictive_native.npz`: E, I, beta and I_obs on 0–199.9 by 0.1 days.
`daily.npz`: the same variables on integer days 0–199, plus 23 analytic
observation-predictive I quantiles for 11 central intervals and the median.
`I_obs` contains seeded Gaussian observation draws, not new posterior particles.

Use daily days 1–159 for proposed fit scoring and 160–199 for forecast scoring.
These masks do not change the inference days 0–159. Synthetic truth remains
separate; Toronto has no known latent E or beta truth. Final WIS and exact
Brandon-code agreement remain pending his scorer.

Case one: use the observed-table `is_observed` mask. Original post-cutoff zero
placeholders are not observations; held-out latent truth is provided separately.


In [ ]:
arrays = {}
for case in selected:
    folder, model, posterior, weights, metadata = load_case(DATA_ROOT, case)
    daily = dict(np.load(OUTPUT / case / "daily.npz"))
    native = dict(np.load(OUTPUT / case / "latent_predictive_native.npz"))
    observations = np.genfromtxt(folder / "observed_daily.csv", delimiter=",", names=True)
    truth = dict(np.load(folder / "generating_curves.npz")) if (folder / "generating_curves.npz").exists() else None
    arrays[case] = dict(posterior=posterior, weights=weights, metadata=metadata,
        native=native, daily=daily, observations=observations, generating_truth=truth,
        fit_mask=(daily["time"] >= 1) & (daily["time"] <= 159),
        forecast_mask=(daily["time"] >= 160) & (daily["time"] <= 199))
    print(case, "posterior", posterior.shape, "daily I", daily["I"].shape,
          "native I", native["I"].shape, "quantiles", daily["predictive_I_quantiles"].shape)

# Example (choose a case):
# d = arrays["case1"]
# d["daily"]["I"][:, d["forecast_mask"]]        # latent I, all 4000 draws
# d["daily"]["I_obs"][:, d["forecast_mask"]]    # seeded observation-predictive draws
# d["daily"]["predictive_I_quantiles"]          # analytic Gaussian mixture, 23 x 200


## Diagnostic figures

Fit/forecast panels use SoS blue, mean trajectories and pointwise equal-tail
latent bands. The I panels additionally show observation-predictive bounds;
these are not simultaneous trajectory intervals. Posterior marginals are saved
separately and are not independent-chain trace plots. Comparator arrays are
not included or invented. See each case's sampler and boundary diagnostics.


In [ ]:
from IPython.display import display, Image
for case in selected:
    display(Image(filename=str(OUTPUT / case / "fit_forecast.png")))
    display(Image(filename=str(OUTPUT / case / "posterior_diagnostic.png")))
